In [ ]:
from datascience import *
import numpy as np
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

In [ ]:
ckd = Table.read_table('ckd.csv')
ckd = ckd.relabeled('Blood Glucose Random', 'Glucose').select('Glucose', 'Hemoglobin', 'White Blood Cell Count', 'Class')

In [ ]:
patients = Table.read_table('breast-cancer.csv').drop('ID')

def randomize_column(a):
    return a + np.random.normal(0.0, 0.09, size=len(a))

jittered = Table().with_columns([
        'Bland Chromatin (jittered)', 
        randomize_column(patients.column('Bland Chromatin')),
        'Single Epithelial Cell Size (jittered)', 
        randomize_column(patients.column('Single Epithelial Cell Size')),
        'Class',
        patients.column('Class')
    ])

# Google Science Fair

Brittany Wenger, a 17-year-old high school student in 2012
 won by building a breast cancer classifier with 99% accuracy. 
 
After imaging, technicians went thru the images and measured certain attributes that helped determine whether the patient had breast cancer. 

* Class of 0: Does not have cancer </br>
* Class of 1: Does have cancer

Import the table and describe: 

In [ ]:
# Import the patient data used to determine they had breast cancer. 

patients = Table.read_table('breast-cancer.csv').drop('ID')
patients.show(5)

In [ ]:
# COMPLETE: Generate a small table that shows have many patients have cancer and how many do not.
# Do not reassign. 



In [ ]:
# COMPLETE: Generate a scatter comparing the Bland Chromatin and Single Epithelial Cell Size grouped by Class.



***QUESTION: Does our graph look like it represents all 680+ patients?***



***QUESTION: Why do you think that is?***


In [ ]:
# Add 'noise' to the data points to reveal hidden values. 

jittered.scatter(0, 1, group='Class')

## Distance ##
Use the distance formula from Pythagoras to determine the distance between two points. 

Recall: Pythagorean Theorem for a right triangle is $a^2 + b^2 = c^2$ 

***QUESTION: How should we rewrite this to find the distance between two points: $(x_1, y_1), (x_2, y_2)$?***

***QUESTION: What would change if the "points" weren't on a 2 dimensional plane (i.e. for any $n$-tuple)?***




In [ ]:
# COMPLETE: Create a function that will return the distance between two points, represented as any n-tuple.

def distance(pt1, pt2):
    """Return the distance between two points, represented as arrays"""
    return 

In [ ]:
# Creates a function that uses the above function to return the distance between two numerical rows of a table.

def row_distance(row1, row2):
    """Return the distance between two numerical rows of a table"""
    return distance( np.array(row1), np.array(row2) )

In [ ]:
# COMPLETE: Create a new table of patients that does not show the Class.

attributes = 
attributes.show(3)

In [ ]:
# COMPLETE: Compare the distance for Row 2 with Row 2. What should we get?



In [ ]:
# COMPLETE: Compare the distance for Row 0 with Row 1



In [ ]:
# COMPLETE: Compare the distance for Row 0 with Row 2



# The Classifier

A classifier uses nearest neighbor predictions to determine the proper class to assign. The neighbors are determined by the distance formula applied to an incoming patient and its closeness to other similar patients (points on the plane). 

We can specify how many points to consider as neighbors. We are calling that value $k$.

The assigned class will be based on the classification of the majority of the $k$ neighbors. 

We can test our classifer by taking a known record and running it through to see if it is correctly classified. 

In [ ]:
def distances(training, example):
    """
    Compute distance between example and every row in training.
    Return training augmented with Distance column
    """
    distances = make_array()
    attributes_only = training.drop('Class')
    
    for row in attributes_only.rows:
        distances = np.append(distances, row_distance(row, example))
    
#   ^ SAME AS DOING:
#
#   for i in np.arange(attributes_only.num_rows):
#       row = attributes_only.row(i)
#       distances = np.append(distances, row_distance(row, example))
        
    return training.with_column('Distance_to_ex', distances)

In [ ]:
# Let Row 21 be our example patient to make sure the classifier is working. 

example = patients.row(21)
example

In [ ]:
# COMPLETE: Reassign example to the row 21 so the class is not included in the information.

example = 
example

In [ ]:
# Measure the distances, importing the table excluding row 21.

distances(patients.exclude(21), example).sort('Distance_to_ex')

In [ ]:
# What does this function do?

def closest(training, example, k):
    """
    Return a table of the k closest neighbors to example
    """
    return distances(training, example).sort('Distance_to_ex').take(np.arange(k))

In [ ]:
# COMPLETE: Run the function with 5 neighbors. 



In [ ]:
closest(patients.exclude(21), example, 5).group('Class').sort('count', descending=True)

***QUESTION: Based on the nearest neighbors, what would patient 21 be classified as? Is that correct?***

In [ ]:
# What does this function do?
def majority_class(topk):
    """
    Return the class with the highest count
    """
    return topk.group('Class').sort('count', descending=True).column(0).item(0)

In [ ]:
# What does this function do? 
def classify(training, example, k):
    """
    Return the majority class among the 
    k nearest neighbors of example
    """
    return majority_class(closest(training, example, k))

In [ ]:
# COMPLETE: Use the classify function with row 21 as the example and 5 nearest neighbors. 




In [ ]:
# COMPLETE: Show row 21 with the class designation.



In [ ]:
# COMPLETE: Assign new_example to row 10 without the class designation. 

new_example = 
new_example

In [ ]:
# COMPLETE: Use the classify function with the new example and 5 nearest neighbors.
# Since the classifier determines the majority class, the training set must include the class as an attribute.



In [ ]:
# COMPLETE: Show row 10 with the class designation. 



In [ ]:
another_example = attributes.row(15)
classify(patients.exclude(15), another_example, 15)

In [ ]:
patients.take(15)

***QUESTION: How many of the above were correctly classified?***



***QUESTION: Would you want to use this to classify new patients? Why or Why not.***



## Review of the Steps for Building a Classifier ##

- `distance(pt1, pt2)`: Returns the distance between the arrays `pt1` and `pt2`
- `row_distance(row1, row2)`: Returns the distance between the rows `row1` and `row2`
- `distances(training, example)`: Returns a table that is `training` with an additional column `'Distance'` that contains the distance between `example` and each row of `training`
- `closest(training, example, k)`: Returns a table of the rows corresponding to the k smallest distances 
- `majority_class(topk)`: Returns the majority class in the `'Class'` column
- `classify(training, example, k)`: Returns the predicted class of `example` based on a `k` nearest neighbors classifier using the historical sample `training`

## Accuracy of a Classifier ##

Create a function that will return the proportion of correctly classified examples of a test set. 

Check the classifier with varying values of $k$.

Data that will be used in this section: 
 * Breast Cancer Data: patients.csv
 * Chronic Kidney Disease Data: ckd.csv

In [ ]:
# COMPLETE: Determine how many patients are in the table of the Breast Cancer Data?



In [ ]:
# COMPLETE: Create a sample from our existing sample that does not allow for duplicates but has the same number of records.

shuffled = patients.sample(with_replacement=False)

# What do these two lines do?

training_set = shuffled.take(np.arange(342))
test_set  = shuffled.take(np.arange(342, 683))

In [ ]:
# What does this function do?

def evaluate_accuracy(training, test, k):
    """Return the proportion of correctly classified examples 
    in the test set"""
    test_attributes = test.drop('Class')
    num_correct = 0
    for i in np.arange(test.num_rows):
        c = classify(training, test_attributes.row(i), k)
        num_correct = num_correct + (c == test.column('Class').item(i))
    return num_correct / test.num_rows

In [ ]:
# COMPLETE: Run the accuracy evaluator with 5 comparisons. 



In [ ]:
# COMPLETE: Run the accuracy evaluator with 3 comparisons. 



In [ ]:
# COMPLETE: Run the accuracy evaluator with 11 comparisons. 



In [ ]:
# COMPLETE: Run the accuracy evaluator with 7 comparisons. 



In [ ]:
# COMPLETE: Run the accuracy evaluator with 9 comparisons. 



In [ ]:
# COMPLETE: Run the accuracy evaluator with 1 comparison. 



***QUESTION: Which value of $k$ gave you the most accurate classifier?***

***QUESTION: If you rerun the cells above, does the same value of $k$ remain the best one?***

***QUESTION: Why are we only using odd numbers for $k$?***



# Standardize Data if Necessary

With the breast cancer data the numbers used to measure attributes of the cells are very similar in range. 

With the CKD data some of the measures are very different.  
For example Glucose ranges from 70-140 mg/dL but Hemoglobin ranges from 8 to 11 grams. 

Standardizing will help very different values to act more similar.

***QUESTION: How does standardizing change our data?***

In [ ]:
# Looking at very different values may skew the classification. So standardization allows comparison to the mean.
ckd.show(3)

# Before Standardizing. 

In [ ]:
# Creates a random sample, training set, test set from the CKD data.

shuffled_ckd = ckd.sample(with_replacement=False) 
training_set_ckd = shuffled.take(np.arange(79))
test_set_ckd  = shuffled.take(np.arange(79, 158))

In [ ]:
# COMPLETE: Check the accuracy before standardization with 5 comparisons. 



In [ ]:
def standard_units(x):
    return (x - np.average(x)) / np.std(x)

In [ ]:
# Create a table with new colums for the standardized values for Glucose, Hemoglobin, and White Blood Cell Count.

ckd_new = ckd.select('Class').with_columns(
    'Glucose_su', standard_units(ckd.column('Glucose')),
    'Hemoglobin_su', standard_units(ckd.column('Hemoglobin')),
    'WBC_su', standard_units(ckd.column('White Blood Cell Count'))
)

In [ ]:
ckd_new
# After Standardizing

In [ ]:
shuffled_ckd_su = ckd_new.sample(with_replacement=False) 
training_set_ckd_su = shuffled.take(np.arange(79))
test_set_ckd_su  = shuffled.take(np.arange(79, 158))

In [ ]:
# COMPLETE: Check the accuracy after standardization with 5 comparisons. 



In [ ]:
# Will it improve the breast cancer classifier? Review the values of the columns. Will standardizing help?

patients.take(15)


In [ ]:
# Creates a new table with all the attributes standardized. 

patients_new = patients.select('Class').with_columns(
    'Clump_Thickness', standard_units(patients.column('Clump Thickness')),
    'Uniformity1', standard_units(patients.column('Uniformity of Cell Size')),
    'Uniformity2', standard_units(patients.column('Uniformity of Cell Shape')),
    'Marginal', standard_units(patients.column('Marginal Adhesion')),
    'Epithelial', standard_units(patients.column('Single Epithelial Cell Size')),
    'Nuclei', standard_units(patients.column('Bare Nuclei')),
    'Chromatin', standard_units(patients.column('Bland Chromatin')),
    'Nucleoli', standard_units(patients.column('Normal Nucleoli')),
    'Mitoses', standard_units(patients.column('Mitoses'))
)
patients_new

In [ ]:
shuffled = patients_new.sample(with_replacement=False) # Randomly permute the rows
training_set = shuffled.take(np.arange(342))
test_set  = shuffled.take(np.arange(342, 683))

In [ ]:
# COMPLETE: Check the accuracy after standardization with 5 comparisons. 


***QUESTION: Was the standardized data better than your best classifier without standarization?***